In [ ]:
!pip install mlflow torchmetrics torch-fidelity

# Importations des librairies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, TensorSpec
import tempfile
import os
import json
import sys

# Montage du drive pour la persistance

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PF1/mlruns/"
os.makedirs(PROJECT_DIR, exist_ok=True)

OUT_DIR = os.path.join(PROJECT_DIR, "runs")
os.makedirs(OUT_DIR, exist_ok=True)

print("Dossier de projet :", PROJECT_DIR)

In [ ]:
import importlib
importlib.invalidate_caches()

# Importer du fichier GAN fourni par le Dev Full Stack

In [ ]:
sys.path.append('/content/drive/MyDrive/PF1/')
from gan import Generator, Discriminator, weights_init, gradient_penalty

# Configuration de MLFLOW

In [ ]:
MLFLOW_LOCAL_DIR = "/content/drive/MyDrive/PF1/mlruns/"
os.makedirs(MLFLOW_LOCAL_DIR, exist_ok=True)

MLFLOW_DB_PATH = os.path.join(MLFLOW_LOCAL_DIR, "mlflow.db")
tracking_uri = f"sqlite:///{MLFLOW_DB_PATH}"
mlflow.set_tracking_uri(tracking_uri)

MLFLOW_EXPERIMENT = "dcgan_vs_wgan_gp_image"
ARTIFACT_LOCATION = f"file://{os.path.join(MLFLOW_LOCAL_DIR, 'artifacts')}"

try:
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if experiment is None:
        mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=ARTIFACT_LOCATION)
except Exception:
    get_ipython().system(f"mlflow db upgrade {tracking_uri}")
    if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT) is None:
        mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=ARTIFACT_LOCATION)

mlflow.set_experiment(MLFLOW_EXPERIMENT)
print("Suivi MLflow configuré avec succès !")

# Configuration Globale

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:{DEVICE}")

In [ ]:
IMG_SIZE = 32
CHANNELS = 1             # 1 pour Fashion-MNIST (niveaux de gris), 3 si CIFAR-10
FEATURE_MAPS = 64        # parametre par defaut de gan.py (largeur des couches conv)
LATENT_DIM = 100
BATCH_SIZE = 128
N_EPOCHS = 20
LR = 0.0002
LR_WGAN = 0.0001
BETA1, BETA2 = 0.5, 0.999

In [ ]:
def set_seed(seed: int):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

def get_dataloader(batch_size: int = BATCH_SIZE) -> DataLoader:
  transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 28x28 -> 32x32
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
  ])
  dataset = datasets.FashionMNIST(
      root="./data", train=True, download=True, transform=transform
  )
  return DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Fonction utilitaire MLflow — log d'échantillons visuels

In [ ]:
def log_sample_grid_to_mlflow(generator, latent_dim, device, epoch, n_samples=16, tag="samples"):
    
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim, device=device)
        imgs = generator(z).cpu().squeeze(1).numpy()
    generator.train()

    cols = int(np.sqrt(n_samples))
    rows = int(np.ceil(n_samples / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.2))
    for idx, ax in enumerate(axes.flat):
        if idx < n_samples:
            ax.imshow((imgs[idx] + 1) / 2, cmap="gray")
        ax.axis("off")
    fig.suptitle(f"Époque {epoch}")

    with tempfile.TemporaryDirectory() as tmp_dir:
        filepath = os.path.join(tmp_dir, f"{tag}_epoch_{epoch:03d}.png")
        fig.savefig(filepath, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(filepath, artifact_path="samples")

# DCGAN - Générateur et Discriminateur

In [ ]:
def train_dcgan(n_epochs=N_EPOCHS, seed=0, verbose=True, log_every=5,
                 nested=False, run_name=None):

    set_seed(seed)
    dataloader = get_dataloader()

    # Instanciation des classes de gan.py :
    # - Discriminator avec use_batchnorm=True, use_sigmoid=True -> variante DCGAN classique
    G = Generator(LATENT_DIM, CHANNELS, FEATURE_MAPS).to(DEVICE)
    D = Discriminator(CHANNELS, FEATURE_MAPS, use_batchnorm=True, use_sigmoid=True).to(DEVICE)

    # Initialisation des poids recommandee par le papier DCGAN (N(0, 0.02)),
    # fournie dans gan.py
    G.apply(weights_init)
    D.apply(weights_init)

    criterion = nn.BCELoss()
    opt_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
    opt_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

    history = {"loss_G": [], "loss_D": []}
    run_name = run_name or f"dcgan_seed{seed}"

    with mlflow.start_run(run_name=run_name, nested=nested) as run:
        # Log des hyperparametres (une seule fois, au debut du run)
        mlflow.log_params({
            "model_type": "DCGAN",
            "seed": seed,
            "n_epochs": n_epochs,
            "batch_size": BATCH_SIZE,
            "latent_dim": LATENT_DIM,
            "channels": CHANNELS,
            "feature_maps": FEATURE_MAPS,
            "img_size": IMG_SIZE,
            "lr": LR,
            "beta1": BETA1,
            "beta2": BETA2,
            "loss_function": "BCE",
        })

        for epoch in range(n_epochs):
            for real_imgs, _ in dataloader:
                real_imgs = real_imgs.to(DEVICE)
                batch_size = real_imgs.size(0)

                # Labels "mous" (soft labels) : 0.9 au lieu de 1.0 pour le vrai,
                # astuce classique qui reduit legerement l'instabilite du DCGAN
                real_labels = torch.full((batch_size,), 0.9, device=DEVICE)
                fake_labels = torch.zeros(batch_size, device=DEVICE)

                # --- Discriminateur ---
                z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
                fake_imgs = G(z).detach()  # pas de gradient dans G ici

                opt_D.zero_grad()
                loss_real = criterion(D(real_imgs), real_labels)
                loss_fake = criterion(D(fake_imgs), fake_labels)
                loss_D = loss_real + loss_fake
                loss_D.backward()
                opt_D.step()

                # --- Générateur ---
                z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
                fake_imgs = G(z)

                opt_G.zero_grad()
                loss_G = criterion(D(fake_imgs), torch.ones(batch_size, device=DEVICE))
                loss_G.backward()
                opt_G.step()

            history["loss_G"].append(loss_G.item())
            history["loss_D"].append(loss_D.item())

            # Log des metriques a chaque epoque (step=epoch -> vraie courbe dans l'UI)
            mlflow.log_metrics({
                "loss_G": loss_G.item(),
                "loss_D": loss_D.item(),
            }, step=epoch)

            # Log periodique d'echantillons visuels
            if (epoch + 1) % log_every == 0 or epoch == n_epochs - 1:
                log_sample_grid_to_mlflow(G, LATENT_DIM, DEVICE, epoch + 1, tag="dcgan")

            if verbose:
                print(f"[DCGAN][seed={seed}] Époque {epoch+1}/{n_epochs} "
                      f"| loss_D={loss_D.item():.4f} | loss_G={loss_G.item():.4f}")

        # Log du modele final entraine, versionne automatiquement par MLflow
        sample_input = torch.randn(1, LATENT_DIM, device=DEVICE)

        input_schema = Schema([
            TensorSpec(np.dtype(np.float32), (-1, LATENT_DIM), name="latent_vector")
        ])

        signature = ModelSignature(inputs=input_schema)

        mlflow.pytorch.log_model(
          G,
          name="generator",
          serialization_format="pt2",
          input_example=sample_input,
          signature=signature,
          pip_requirements=["torch==2.11.0+cu128", "torchvision"]
        )
        mlflow.pytorch.log_model(
            D,
            name="discriminator",
            serialization_format="pickle"
        )
        history["mlflow_run_id"] = run.info.run_id

    return G, D, history

# WGAN-GP - Générateur, Critique et Gradient Penalty

In [ ]:
def train_wgan_gp(n_epochs=N_EPOCHS, seed=0, n_critic=5, lambda_gp=10.0,
                   verbose=True, log_every=5, nested=False, run_name=None):
    
    set_seed(seed)
    dataloader = get_dataloader()

    # Meme classe Generator que pour le DCGAN (architecture volontairement
    # identique, seule la loss differe entre les deux variantes)
    G = Generator(LATENT_DIM, CHANNELS, FEATURE_MAPS).to(DEVICE)
    # Discriminator utilise ici comme CRITIQUE : use_batchnorm=False (remplace
    # par InstanceNorm en interne), use_sigmoid=False (sortie reelle non bornee)
    C = Discriminator(CHANNELS, FEATURE_MAPS, use_batchnorm=False, use_sigmoid=False).to(DEVICE)

    G.apply(weights_init)
    C.apply(weights_init)

    # Adam avec beta1=0 recommande dans le papier WGAN-GP original
    opt_G = optim.Adam(G.parameters(), lr=LR_WGAN, betas=(0.0, 0.9))
    opt_C = optim.Adam(C.parameters(), lr=LR_WGAN, betas=(0.0, 0.9))

    history = {"loss_G": [], "loss_D": []}
    run_name = run_name or f"wgan_gp_seed{seed}"

    with mlflow.start_run(run_name=run_name, nested=nested) as run:
        mlflow.log_params({
            "model_type": "WGAN-GP",
            "seed": seed,
            "n_epochs": n_epochs,
            "batch_size": BATCH_SIZE,
            "latent_dim": LATENT_DIM,
            "channels": CHANNELS,
            "feature_maps": FEATURE_MAPS,
            "img_size": IMG_SIZE,
            "lr": LR_WGAN,
            "n_critic": n_critic,
            "lambda_gp": lambda_gp,
            "loss_function": "Wasserstein + Gradient Penalty",
        })

        for epoch in range(n_epochs):
            for i, (real_imgs, _) in enumerate(dataloader):
                real_imgs = real_imgs.to(DEVICE)
                batch_size = real_imgs.size(0)

                # --- Critique ---
                z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
                fake_imgs = G(z).detach()

                opt_C.zero_grad()
                gp = gradient_penalty(C, real_imgs, fake_imgs, DEVICE)
                loss_C = -torch.mean(C(real_imgs)) + torch.mean(C(fake_imgs)) + lambda_gp * gp
                loss_C.backward()
                opt_C.step()

                # --- Générateur ---
                if i % n_critic == 0:
                    z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
                    fake_imgs = G(z)

                    opt_G.zero_grad()
                    loss_G = -torch.mean(C(fake_imgs))
                    loss_G.backward()
                    opt_G.step()

            history["loss_G"].append(loss_G.item())
            history["loss_D"].append(loss_C.item())

            mlflow.log_metrics({
                "loss_G": loss_G.item(),
                "loss_C": loss_C.item(),
                "gradient_penalty": gp.item(),
            }, step=epoch)

            if (epoch + 1) % log_every == 0 or epoch == n_epochs - 1:
                log_sample_grid_to_mlflow(G, LATENT_DIM, DEVICE, epoch + 1, tag="wgan_gp")

            if verbose:
                print(f"[WGAN-GP][seed={seed}] Époque {epoch+1}/{n_epochs} "
                      f"| loss_C={loss_C.item():.4f} | loss_G={loss_G.item():.4f}")

        mlflow.pytorch.log_model(G, artifact_path="generator", serialization_format="pickle")
        mlflow.pytorch.log_model(C, artifact_path="critic", serialization_format="pickle")
        history["mlflow_run_id"] = run.info.run_id

    return G, C, history

# Détection de mode collapse / divergence

In [ ]:
def detect_divergence(history, threshold=50.0):
    
    loss_g = np.array(history["loss_G"])
    loss_d = np.array(history["loss_D"])
    has_nan = np.isnan(loss_g).any() or np.isnan(loss_d).any()
    has_explosion = (np.abs(loss_g) > threshold).any() or (np.abs(loss_d) > threshold).any()
    return bool(has_nan or has_explosion)


def detect_mode_collapse(generator, latent_dim, device, n_samples=200, pixel_std_threshold=0.02):
    
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim, device=device)
        samples = generator(z).view(n_samples, -1)
    pixel_std = samples.std(dim=0).mean().item()
    generator.train()
    return pixel_std < pixel_std_threshold, pixel_std


def evaluate_run_stability(model_name, generator, history, latent_dim, device, log_to_mlflow_run=None):
    
    diverged = detect_divergence(history)
    collapsed, pixel_std = detect_mode_collapse(generator, latent_dim, device)
    converged = not diverged and not collapsed

    result = {
        "model": model_name, "diverged": diverged, "mode_collapse": collapsed,
        "pixel_std": pixel_std, "converged": converged
    }

    if log_to_mlflow_run is not None:
        with mlflow.start_run(run_id=log_to_mlflow_run, nested=True):
            mlflow.log_metric("pixel_std_final", pixel_std)
            mlflow.set_tags({
                "diverged": str(diverged),
                "mode_collapse": str(collapsed),
                "converged": str(converged),
            })

    return result

# Diversité

In [ ]:
def compute_generation_diversity(generator, latent_dim, device, n_samples=200):
    
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim, device=device)
        samples = generator(z).view(n_samples, -1) # Aplatissement des images
        
    generator.train()
    
    # Calcul de la matrice des distances euclidiennes par paires
    # torch.cdist calcule la distance p-norm entre chaque paire de lignes
    dist_matrix = torch.cdist(samples, samples, p=2.0)
    
    # On exclue la diagonale (distance d'un échantillon avec lui-même = 0)
    mask = ~torch.eye(n_samples, dtype=torch.bool, device=device)
    mean_distance = dist_matrix[mask].mean().item()
    
    return mean_distance

# Étude d'ablation

In [ ]:
ABLATION_SEEDS = [42, 123, 7]


def run_ablation_study(seeds=ABLATION_SEEDS, n_epochs=30):
    
    results = []

    with mlflow.start_run(run_name="ablation_study") as parent_run:
        mlflow.log_params({"seeds": seeds, "n_epochs_per_run": n_epochs})

        for seed in seeds:
            print(f"\n=== Seed {seed} ===")

            # --- DCGAN (run enfant imbrique) ---
            G_d, D_d, hist_d = train_dcgan(
                n_epochs=n_epochs, seed=seed, verbose=False,
                nested=True, run_name=f"dcgan_seed{seed}"
            )
            eval_d = evaluate_run_stability(
                "DCGAN", G_d, hist_d, LATENT_DIM, DEVICE,
                log_to_mlflow_run=hist_d["mlflow_run_id"]
            )
            eval_d["seed"] = seed
            results.append(eval_d)

            # --- WGAN-GP (run enfant imbrique) ---
            G_w, C_w, hist_w = train_wgan_gp(
                n_epochs=n_epochs, seed=seed, verbose=False,
                nested=True, run_name=f"wgan_gp_seed{seed}"
            )
            eval_w = evaluate_run_stability(
                "WGAN-GP", G_w, hist_w, LATENT_DIM, DEVICE,
                log_to_mlflow_run=hist_w["mlflow_run_id"]
            )
            eval_w["seed"] = seed
            results.append(eval_w)

        df = pd.DataFrame(results)

        # Tableau de stabilite final (livrable demande)
        stability_table = df.groupby("model").agg(
            runs_total=("converged", "count"),
            runs_converges=("converged", "sum"),
            runs_diverged=("diverged", "sum"),
            runs_collapsed=("mode_collapse", "sum"),
        ).reset_index()
        stability_table["taux_convergence_%"] = (
            stability_table["runs_converges"] / stability_table["runs_total"] * 100
        )

        # Log du tableau recapitulatif au niveau du run PARENT
        for _, row in stability_table.iterrows():
            model_tag = row["model"].lower().replace("-", "_")
            mlflow.log_metric(f"{model_tag}_taux_convergence_pct", row["taux_convergence_%"])
            mlflow.log_metric(f"{model_tag}_runs_diverged", row["runs_diverged"])
            mlflow.log_metric(f"{model_tag}_runs_collapsed", row["runs_collapsed"])

        with tempfile.TemporaryDirectory() as tmp_dir:
            detail_path = os.path.join(tmp_dir, "ablation_detail.csv")
            table_path = os.path.join(tmp_dir, "tableau_stabilite.csv")
            df.to_csv(detail_path, index=False)
            stability_table.to_csv(table_path, index=False)
            mlflow.log_artifact(detail_path)
            mlflow.log_artifact(table_path)

    return df, stability_table

# Evaluation FID

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance


def compute_fid(generator, dataloader, latent_dim, device, n_images=1000):
    
    fid = FrechetInceptionDistance(feature=64, normalize=False).to(device)

    n_collected = 0
    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        real_imgs = ((real_imgs + 1) / 2 * 255).to(torch.uint8)
        real_imgs = real_imgs.repeat(1, 3, 1, 1)
        fid.update(real_imgs, real=True)
        n_collected += real_imgs.size(0)
        if n_collected >= n_images:
            break

    generator.eval()
    with torch.no_grad():
        n_generated = 0
        while n_generated < n_images:
            batch = min(128, n_images - n_generated)
            z = torch.randn(batch, latent_dim, device=device)
            fake_imgs = generator(z)
            fake_imgs = ((fake_imgs + 1) / 2 * 255).to(torch.uint8)
            fake_imgs = fake_imgs.repeat(1, 3, 1, 1)
            fid.update(fake_imgs, real=False)
            n_generated += batch
    generator.train()

    return fid.compute().item()


def evaluate_final_models(G_dcgan, G_wgan):
    
    dataloader = get_dataloader()
    fid_dcgan, fid_wgan = None, None

    with mlflow.start_run(run_name="evaluation_finale_dcgan"):
        try:
            fid_dcgan = compute_fid(G_dcgan, dataloader, LATENT_DIM, DEVICE)
            mlflow.log_metric("fid_dcgan", fid_dcgan)
            print(f"FID DCGAN   : {fid_dcgan:.4f}")
        except Exception as e:
            mlflow.set_tag("fid_error", str(e))
            print(f"FID DCGAN non calcule ({e}) - run cree mais sans metrique fid_dcgan.")

    with mlflow.start_run(run_name="evaluation_finale_wgan_gp"):
        try:
            fid_wgan = compute_fid(G_wgan, dataloader, LATENT_DIM, DEVICE)
            mlflow.log_metric("fid_wgan_gp", fid_wgan)
            print(f"FID WGAN-GP : {fid_wgan:.4f}")
        except Exception as e:
            mlflow.set_tag("fid_error", str(e))
            print(f"FID WGAN-GP non calcule ({e}) - run cree mais sans metrique fid_wgan_gp.")

    return fid_dcgan, fid_wgan

# Charger un modèle entraîné depuis MLflow (pour le Backend)

In [ ]:
import mlflow.pytorch

def load_generator_from_mlflow(run_id: str, artifact_path: str = "generator"):
    """Charge un generateur PyTorch directement depuis un run MLflow."""
    model_uri = f"runs:/{run_id}/{artifact_path}"
    model = mlflow.pytorch.load_model(model_uri)
    model.eval()
    return model

# Export vers le Backend - format strict convenu

In [ ]:
import uuid

def generate_run_id(model_type: str, dataset: str, seed: int) -> str:
    suffix = uuid.uuid4().hex[:4]
    return f"{model_type}_{dataset}_seed{seed}_{suffix}"


def export_run_for_backend(generator, output_root, model_type, dataset,
                            latent_dim, channels, seed, epochs_trained,
                            converged=None, mode_collapse_detected=False,
                            mlflow_run_id=None):
    
    run_id = generate_run_id(model_type, dataset, seed)
    run_dir = os.path.join(output_root, run_id)
    os.makedirs(run_dir, exist_ok=True)

    weights_path = os.path.join(run_dir, "generator.pt")
    torch.save(generator.state_dict(), weights_path)

    config = {
        "run_id": run_id,
        "model_type": model_type,
        "dataset": dataset,
        "latent_dim": latent_dim,
        "channels": channels,
        "seed": seed,
        "epochs_trained": epochs_trained,
        "mode_collapse_detected": mode_collapse_detected,
    }
    if converged is not None:
        config["converged"] = converged

    config_path = os.path.join(run_dir, "config.json")
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    if mlflow_run_id is not None:
        with mlflow.start_run(run_id=mlflow_run_id, nested=True):
            mlflow.log_artifact(weights_path, artifact_path="backend_export")
            mlflow.log_artifact(config_path, artifact_path="backend_export")
            mlflow.set_tag("backend_run_id", run_id)
            mlflow.set_tag("backend_export_path", run_dir)
    else:
        print("[*]  mlflow_run_id non fourni : ce run n'apparaitra PAS dans MLflow "
              "(uniquement sur Drive, dans OUT_DIR).")

    print(f"Export termine : {run_dir}")
    print(f"  - {weights_path}")
    print(f"  - {config_path}")

    return run_dir


ALLOWED_FIELDS = {
    "run_id", "model_type", "dataset", "latent_dim", "channels",
    "seed", "epochs_trained", "converged", "mode_collapse_detected",
}
REQUIRED_FIELDS = ALLOWED_FIELDS - {"converged"}


def validate_export(run_dir):
    
    # Verification stricte AVANT de livrer un dossier au Backend 
    weights_path = os.path.join(run_dir, "generator.pt")
    config_path = os.path.join(run_dir, "config.json")

    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"generator.pt manquant dans {run_dir}")
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"config.json manquant dans {run_dir}")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    folder_name = os.path.basename(os.path.normpath(run_dir))
    if config.get("run_id") != folder_name:
        raise ValueError(
            f"run_id ('{config.get('run_id')}') != nom du dossier ('{folder_name}')"
        )

    present_fields = set(config.keys())
    missing = REQUIRED_FIELDS - present_fields
    if missing:
        raise ValueError(f"Champs manquants dans config.json : {missing}")

    extra = present_fields - ALLOWED_FIELDS
    if extra:
        raise ValueError(f"Champs en trop dans config.json (non autorises) : {extra}")

    if config["model_type"] not in ("dcgan", "wgan_gp"):
        raise ValueError(f"model_type invalide : {config['model_type']}")
    if config["dataset"] not in ("fashion_mnist", "cifar10"):
        raise ValueError(f"dataset invalide : {config['dataset']}")
    if config["dataset"] == "fashion_mnist" and config["channels"] != 1:
        raise ValueError("channels doit etre 1 pour fashion_mnist")
    if config["dataset"] == "cifar10" and config["channels"] != 3:
        raise ValueError("channels doit etre 3 pour cifar10")

    print(f"[*] Export valide : {run_dir}")
    return True

# Script principal (orchestration complète)

In [ ]:
if __name__ == "__main__":
    ABLATION_SEEDS = [42, 123, 7]
    results = []
    dcgan_models_hist = {}
    wgan_models_hist = {}

    # Run parent MLflow regroupant les 6 runs enfants (nested=True), pour
    # garder la meme hierarchie de visualisation que precedemment dans l'UI.
    with mlflow.start_run(run_name="ablation_study") as parent_run:
        mlflow.log_params({"seeds": ABLATION_SEEDS, "n_epochs_per_run": 20})

        # Dataloader reutilise pour tous les calculs de FID (evite de
        # recharger Fashion-MNIST 6 fois inutilement)
        fid_dataloader = get_dataloader()

        print(">>> Entraînement DCGAN (3 seeds, 20 époques)")
        for seed_val in ABLATION_SEEDS:
            G, C, hist = train_wgan_gp(n_epochs=30, seed=seed_val,
                                         nested=True, run_name=f"wgan_gp_seed{seed_val}")
            wgan_models_hist[seed_val] = (G, C, hist)

            eval_w = evaluate_run_stability("WGAN-GP", G, hist, LATENT_DIM, DEVICE,
                                             log_to_mlflow_run=hist["mlflow_run_id"])
            eval_w["seed"] = seed_val

            # Calcul du FID
            fid_value = compute_fid(G, fid_dataloader, LATENT_DIM, DEVICE)
            eval_w["fid"] = fid_value
            
            # --- NOUVEAU : Calcul de la diversité ---
            diversity_value = compute_generation_diversity(G, LATENT_DIM, DEVICE)
            eval_w["diversity"] = diversity_value

            with mlflow.start_run(run_id=hist["mlflow_run_id"], nested=True):
                mlflow.log_metric("fid", fid_value)
                mlflow.log_metric("diversity", diversity_value) # Log MLflow

            results.append(eval_w)

        print(">>> Entraînement WGAN-GP (3 seeds, 20 époques)")
        for seed_val in ABLATION_SEEDS:
            G, C, hist = train_wgan_gp(n_epochs=20, seed=seed_val,
                                         nested=True, run_name=f"wgan_gp_seed{seed_val}")
            wgan_models_hist[seed_val] = (G, C, hist)

            eval_w = evaluate_run_stability("WGAN-GP", G, hist, LATENT_DIM, DEVICE,
                                             log_to_mlflow_run=hist["mlflow_run_id"])
            eval_w["seed"] = seed_val

            fid_value = compute_fid(G, fid_dataloader, LATENT_DIM, DEVICE)
            eval_w["fid"] = fid_value
            with mlflow.start_run(run_id=hist["mlflow_run_id"], nested=True):
                mlflow.log_metric("fid", fid_value)

            results.append(eval_w)

        # Tableau detaille (une ligne par run : Modele, Seed, Converge,
        # Mode collapse, FID) -> alimente le tableau "Detail des 6 runs" du rapport
        df = pd.DataFrame(results)

        # Tableau agrege par modele -> alimente le "Taux de convergence" et
        # le "FID moyen" du rapport
        tableau_stabilite = df.groupby("model").agg(
            runs_total=("converged", "count"),
            runs_converges=("converged", "sum"),
            runs_diverged=("diverged", "sum"),
            runs_collapsed=("mode_collapse", "sum"),
            fid_moyen=("fid", "mean"),
        ).reset_index()
        tableau_stabilite["taux_convergence_%"] = (
            tableau_stabilite["runs_converges"] / tableau_stabilite["runs_total"] * 100
        )

        print(df)
        print(tableau_stabilite)

        # Log des metriques agregees + artefacts CSV sur le run PARENT
        for _, row in tableau_stabilite.iterrows():
            model_tag = row["model"].lower().replace("-", "_")
            mlflow.log_metric(f"{model_tag}_taux_convergence_pct", row["taux_convergence_%"])
            mlflow.log_metric(f"{model_tag}_runs_diverged", row["runs_diverged"])
            mlflow.log_metric(f"{model_tag}_runs_collapsed", row["runs_collapsed"])
            mlflow.log_metric(f"{model_tag}_fid_moyen", row["fid_moyen"])

        with tempfile.TemporaryDirectory() as tmp_dir:
            detail_path = os.path.join(tmp_dir, "ablation_detail.csv")
            table_path = os.path.join(tmp_dir, "tableau_stabilite.csv")
            df.to_csv(detail_path, index=False)
            tableau_stabilite.to_csv(table_path, index=False)
            mlflow.log_artifact(detail_path)
            mlflow.log_artifact(table_path)

    
    print("\n>>> Export vers le Backend (6 modeles)")

    for seed_val in ABLATION_SEEDS:
        G_dcgan, D_dcgan, hist_dcgan = dcgan_models_hist[seed_val]
        stab_dcgan = evaluate_run_stability("DCGAN", G_dcgan, hist_dcgan, LATENT_DIM, DEVICE)
        run_dir_dcgan = export_run_for_backend(
            generator=G_dcgan, output_root=OUT_DIR, model_type="dcgan",
            dataset="fashion_mnist", latent_dim=LATENT_DIM, channels=CHANNELS,
            seed=seed_val, epochs_trained=20,
            converged=stab_dcgan["converged"], mode_collapse_detected=stab_dcgan["mode_collapse"],
            mlflow_run_id=hist_dcgan["mlflow_run_id"],
        )
        validate_export(run_dir_dcgan)

        G_wgan, C_wgan, hist_wgan = wgan_models_hist[seed_val]
        stab_wgan = evaluate_run_stability("WGAN-GP", G_wgan, hist_wgan, LATENT_DIM, DEVICE)
        run_dir_wgan = export_run_for_backend(
            generator=G_wgan, output_root=OUT_DIR, model_type="wgan_gp",
            dataset="fashion_mnist", latent_dim=LATENT_DIM, channels=CHANNELS,
            seed=seed_val, epochs_trained=20,
            converged=stab_wgan["converged"], mode_collapse_detected=stab_wgan["mode_collapse"],
            mlflow_run_id=hist_wgan["mlflow_run_id"],
        )
        validate_export(run_dir_wgan)

    print(f"\n6 dossiers prets a livrer au Backend : {OUT_DIR}")
    print("Egalement visibles dans MLflow UI, sous chaque run -> Artifacts -> backend_export/")